# T1 + T2. Canonical PM2.5-O3 mechanism sample

T1 + T2 (TODO_paper_figures_and_analyses.md): one canonical Bogota sample for every
PM2.5-O3 mechanism number quoted in the paper.

Sample: Bogota daily, rows where pm25, o3, no2, no, co, temp_2m, rh_2m, blh_mean,
radiation_mean and fire_eq1 (east-sector lag 2 + north-sector hotspots) are all present (TROPOMI NOT required).
Method: residual-on-residual OLS partial correlation, with intercept.

Outputs (Paper/figures/data/):
  mechanism_attenuation_canonical.csv   control, partial_r, ci_lo, ci_hi, pct_attenuation, n
  temperature_terciles_canonical.csv    Paso-2 style tercile test on temp_2m
  conditioning_terciles_canonical.csv   same test, conditioning on fire/RH/w700/radiation/stagnation
  canonical_sample_note.txt

In [1]:
# Setup
import sys
from pathlib import Path
HERE = Path.cwd().resolve()          # notebooks/ folder
sys.path.insert(0, str(HERE.parent)); sys.path.insert(0, str(HERE.parents[1]))
from pathlib import Path

import numpy as np
import pandas as pd
from style import *   # PM25, O3, NAVY, GREY, RED, LIGHT, ROOT, DATA_OUT, save_fig, panel_label

In [2]:
# Analysis
ROOT = ROOT
OUT = ROOT / "Paper" / "figures" / "data"
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(ROOT / "Data/processed/colombia/bogota/bogota_o3_master_dataset.parquet")
df["date"] = pd.to_datetime(df["date"])
# 2026-10-20: "fire" in every control and conditioner is the fire quantity of the PM2.5 lead equation (Eq. 1) after the fire-file
# correction: fire_eq1 = east-sector hotspots two days earlier + north-sector hotspots the same day (was east sector, lag 3)
df["fire_eq1"] = df["hotspots_sector_E_lag2"] + df["hotspots_sector_N"]

REQUIRED = ["pm25", "o3", "no2", "no", "co", "temp_2m", "rh_2m", "blh_mean",
            "radiation_mean", "fire_eq1"]
d = df.dropna(subset=REQUIRED).reset_index(drop=True)
N = len(d)


def _resid(y, X):
    A = np.column_stack([np.ones(len(X)), X]) if X is not None and X.shape[1] else np.ones((len(y), 1))
    beta, *_ = np.linalg.lstsq(A, y, rcond=None)
    return y - A @ beta


def partial_corr(data, x, y, controls):
    xv = data[x].to_numpy(float)
    yv = data[y].to_numpy(float)
    X = data[controls].to_numpy(float) if controls else None
    rx, ry = _resid(xv, X), _resid(yv, X)
    r = float(np.corrcoef(rx, ry)[0, 1])
    n = len(data)
    k = len(controls)
    z = np.arctanh(r)
    se = 1.0 / np.sqrt(max(n - k - 3, 1))
    return r, float(np.tanh(z - 1.96 * se)), float(np.tanh(z + 1.96 * se)), n


# T1 + T2: attenuation table

In [3]:
# Settings (edit here)
BLOCK = 30        # moving-block bootstrap: block length in days (days are autocorrelated, lag-1 ~ 0.75)
NBOOT = 2000      # bootstrap replicates
SEED = 1

In [4]:
# Moving-block bootstrap (review item R2)
# The Fisher-z intervals treat every day as independent and are therefore too narrow. The intervals reported in the paper
# (columns ci_lo / ci_hi) are 95 % percentile intervals of a moving-block bootstrap; the Fisher-z ones are kept as *_fisher.
def block_resamples(n, block=BLOCK, nboot=NBOOT, seed=SEED):
    rng = np.random.default_rng(seed); nb = int(np.ceil(n / block))
    starts = rng.integers(0, n - block + 1, size=(nboot, nb))
    return (starts[:, :, None] + np.arange(block)).reshape(nboot, -1)[:, :n]

def pcorr(x, y, C=None):
    A = np.column_stack([np.ones(len(x)), C]) if C is not None and C.shape[1] else np.ones((len(x), 1))
    rx = x - A @ np.linalg.lstsq(A, x, rcond=None)[0]; ry = y - A @ np.linalg.lstsq(A, y, rcond=None)[0]
    return float(np.corrcoef(rx, ry)[0, 1])

IDX = block_resamples(N)
xv, yv = d["pm25"].to_numpy(float), d["o3"].to_numpy(float)
_ac = np.corrcoef(yv[:-1], yv[1:])[0, 1]
print(f"lag-1 autocorrelation of O3 = {_ac:.2f}  ->  effective N roughly {N * (1 - _ac) / (1 + _ac):.0f} of {N} (AR(1) approximation)")

lag-1 autocorrelation of O3 = 0.77  ->  effective N roughly 385 of 2917 (AR(1) approximation)


In [5]:
# Attenuation table (T1 + T2)
raw_r, raw_lo, raw_hi, _ = partial_corr(d, "pm25", "o3", [])
raw_b = np.array([pcorr(xv[i], yv[i]) for i in IDX])

CONTROLS = [
    ("NO2 (ground)", ["no2"], "T1"),
    ("fire (Eq. 1 fire term)", ["fire_eq1"], "T1"),
    ("NO2 + fire", ["no2", "fire_eq1"], "T1"),
    ("photochemical_potential + no_ratio", ["photochemical_potential", "no_ratio"], "T1"),
    ("RH (rh_2m)", ["rh_2m"], "T1"),
    ("BLH (blh_mean)", ["blh_mean"], "T1"),
    ("CO", ["co"], "T1"),
    ("temperature (temp_2m)", ["temp_2m"], "T2"),
    ("temperature + NO2 + fire", ["temp_2m", "no2", "fire_eq1"], "T2"),
    ("temperature + radiation", ["temp_2m", "radiation_mean"], "T2"),
]
_lo, _hi = np.percentile(raw_b, [2.5, 97.5])
rows = [dict(control="none (raw corr)", controls="", task="T1", partial_r=raw_r, ci_lo=_lo, ci_hi=_hi, ci_lo_fisher=raw_lo, ci_hi_fisher=raw_hi,
             pct_attenuation=0.0, pct_ci_lo=0.0, pct_ci_hi=0.0, attenuation_ci_excludes_zero=False, n=N)]
for name, ctrl, task in CONTROLS:
    sub = d.dropna(subset=ctrl)
    r, lo, hi, n = partial_corr(sub, "pm25", "o3", ctrl)
    C = d[ctrl].to_numpy(float)
    part_b = np.array([pcorr(xv[i], yv[i], C[i]) for i in IDX])         # same resamples as raw_b -> paired
    pct_b = 100.0 * (raw_b - part_b) / raw_b
    b_lo, b_hi = np.percentile(part_b, [2.5, 97.5]); p_lo, p_hi = np.percentile(pct_b, [2.5, 97.5])
    rows.append(dict(control=name, controls="+".join(ctrl), task=task, partial_r=r, ci_lo=b_lo, ci_hi=b_hi, ci_lo_fisher=lo, ci_hi_fisher=hi,
                     pct_attenuation=100.0 * (raw_r - r) / raw_r, pct_ci_lo=p_lo, pct_ci_hi=p_hi,
                     attenuation_ci_excludes_zero=bool(p_lo > 0 or p_hi < 0), n=n))
att = pd.DataFrame(rows)
att.to_csv(OUT / "mechanism_attenuation_canonical.csv", index=False)
print(att.round(4).to_string(index=False))

# T2: tercile conditioning (Paso-2 protocol)
#   interactive term = pm25 * radiation_mean (raw) and pm25_net * radiation_mean, where
#   pm25_net = residual of pm25 regressed on co (the CO-uncorrelated, non-combustion part).
#   partial corr with o3 controlling for no2 + no, within each tercile of the conditioner.

                           control                         controls task  partial_r  ci_lo  ci_hi  ci_lo_fisher  ci_hi_fisher  pct_attenuation  pct_ci_lo  pct_ci_hi  attenuation_ci_excludes_zero    n
                   none (raw corr)                                    T1     0.4926 0.4005 0.5786        0.4646        0.5196           0.0000     0.0000     0.0000                         False 2917
                      NO2 (ground)                              no2   T1     0.4157 0.3040 0.5150        0.3852        0.4453          15.6044     8.1409    27.8256                          True 2917
            fire (Eq. 1 fire term)                         fire_eq1   T1     0.2939 0.2148 0.3679        0.2604        0.3267          40.3244    31.6617    50.7095                          True 2917
                        NO2 + fire                     no2+fire_eq1   T1     0.1638 0.0780 0.2481        0.1283        0.1990          66.7362    54.9643    81.9510                          True 2917


In [6]:
# Tercile conditioning (Paso-2 protocol)
A = np.column_stack([np.ones(N), d["co"].to_numpy(float)])
beta, *_ = np.linalg.lstsq(A, d["pm25"].to_numpy(float), rcond=None)
d["pm25_net"] = d["pm25"].to_numpy(float) - A @ beta
pm25_co_r2 = 1 - np.var(d["pm25_net"]) / np.var(d["pm25"])
d["int_raw"] = d["pm25"] * d["radiation_mean"]
d["int_net"] = d["pm25_net"] * d["radiation_mean"]

CONTROL_SETS = {
    "no2+no": ["no2", "no"],
    # control set used for the earlier RH/fire/radiation tercile results in MASTER.md
    "legacy w700+lts+day_of_week": ["w700", "lts", "day_of_week"],
}
CONDITIONERS = {
    "temp_2m": "temperature",
    "fire_eq1": "fire (Eq. 1 fire term)",
    "rh_2m": "relative humidity",
    "w700": "w700 (low = ascent, high = subsidence)",
    "radiation_mean": "radiation",
    "wind_speed": "wind speed (low tercile = stagnant)",
}
cond_rows, trend_rows = [], []
for set_name, CTRL in CONTROL_SETS.items():
    for col, label in CONDITIONERS.items():
        if col not in d.columns or any(c not in d.columns for c in CTRL):
            continue
        sub_all = d.dropna(subset=[col] + CTRL)
        q = sub_all[col].rank(method="first")
        tercile = pd.qcut(q, 3, labels=["low", "mid", "high"])
        # block bootstrap over the whole conditioned sample; tercile membership stays attached to each day (fixed thresholds)
        lab = tercile.map({"low": 0, "mid": 1, "high": 2}).to_numpy(int); Cm = sub_all[CTRL].to_numpy(float); yy = sub_all["o3"].to_numpy(float)
        IDXs = block_resamples(len(sub_all))
        boots = {}
        for term in ("int_net", "int_raw", "pm25"):
            xx = sub_all[term].to_numpy(float); out = np.full((len(IDXs), 3), np.nan)
            for bi, i in enumerate(IDXs):
                for k in range(3):
                    ii = i[lab[i] == k]
                    if len(ii) > 30: out[bi, k] = pcorr(xx[ii], yy[ii], Cm[ii])
            boots[term] = out
        for k, t in enumerate(["low", "mid", "high"]):
            sub = sub_all[tercile == t]
            row = dict(control_set=set_name, conditioner=col, label=label, tercile=t,
                       conditioner_mean=float(sub[col].mean()), n=len(sub))
            for term, tag in [("int_net", "net"), ("int_raw", "raw"), ("pm25", "pm25_only")]:
                r, lo, hi, _ = partial_corr(sub, term, "o3", CTRL)
                row[f"partial_r_{tag}"] = r
                row[f"ci_lo_{tag}"], row[f"ci_hi_{tag}"] = np.nanpercentile(boots[term][:, k], [2.5, 97.5])
                row[f"ci_lo_{tag}_fisher"], row[f"ci_hi_{tag}_fisher"] = lo, hi
            cond_rows.append(row)
        for term, tag in [("int_net", "net"), ("int_raw", "raw"), ("pm25", "pm25_only")]:
            dd = boots[term][:, 2] - boots[term][:, 0]                        # high tercile minus low tercile, paired within each resample
            obs = [r_ for r_ in cond_rows if r_["control_set"] == set_name and r_["conditioner"] == col]
            diff = obs[-1][f"partial_r_{tag}"] - obs[-3][f"partial_r_{tag}"]
            lo_, hi_ = np.nanpercentile(dd, [2.5, 97.5])
            trend_rows.append(dict(control_set=set_name, conditioner=col, quantity=tag, high_minus_low=diff, ci_lo=lo_, ci_hi=hi_, excludes_zero=bool(lo_ > 0 or hi_ < 0)))
cond = pd.DataFrame(cond_rows)
trend = pd.DataFrame(trend_rows); trend.to_csv(OUT / "tercile_trend_tests_canonical.csv", index=False)   # does the rise from the low to the high tercile exclude zero?
cond.to_csv(OUT / "conditioning_terciles_canonical.csv", index=False)
cond[(cond.conditioner == "temp_2m") & (cond.control_set == "no2+no")].to_csv(OUT / "temperature_terciles_canonical.csv", index=False)
print()
print(cond.round(3).to_string(index=False))


                control_set    conditioner                                  label tercile  conditioner_mean   n  partial_r_net  ci_lo_net  ci_hi_net  ci_lo_net_fisher  ci_hi_net_fisher  partial_r_raw  ci_lo_raw  ci_hi_raw  ci_lo_raw_fisher  ci_hi_raw_fisher  partial_r_pm25_only  ci_lo_pm25_only  ci_hi_pm25_only  ci_lo_pm25_only_fisher  ci_hi_pm25_only_fisher
                     no2+no        temp_2m                            temperature     low            13.823 972          0.390      0.278      0.497             0.335             0.442          0.561      0.492      0.627             0.517             0.603                0.350            0.249            0.447                   0.294                   0.404
                     no2+no        temp_2m                            temperature     mid            14.689 972          0.518      0.391      0.614             0.471             0.563          0.639      0.551      0.709             0.601             0.675                0.53

In [7]:
# Sample note
note = f"""Canonical sample (T1/T2), written by Scripts/paper/notebooks/T1_T2_canonical_mechanisms.ipynb
Source: Data/processed/colombia/bogota/bogota_o3_master_dataset.parquet (Bogota, daily)
Rows kept: those with all of {REQUIRED} present.
N = {N}; dates {d['date'].min().date()} to {d['date'].max().date()}.
Raw corr(pm25, o3) = {raw_r:.4f}  (95% CI {raw_lo:.3f} to {raw_hi:.3f}).
Method: residual-on-residual OLS partial correlation with intercept. Confidence intervals (ci_lo / ci_hi) are 95 % percentile
intervals of a moving-block bootstrap ({BLOCK}-day blocks, {NBOOT} replicates, seed {SEED}); the attenuation intervals are paired
(same resamples for the raw and the controlled correlation). Fisher-z intervals (days treated as independent, too narrow given
lag-1 autocorrelation ~0.75) are kept as ci_*_fisher.
pm25_net = residual of pm25 ~ co (R2 of that regression on this sample = {pm25_co_r2:.3f}).
Tercile test: partial corr(pm25*radiation_mean [or pm25_net*radiation_mean], o3 | no2 + no)
within terciles of each conditioner; terciles by rank so they are equal-sized.
"""
(OUT / "canonical_sample_note.txt").write_text(note)
print(note)

Canonical sample (T1/T2), written by Scripts/paper/notebooks/T1_T2_canonical_mechanisms.ipynb
Source: Data/processed/colombia/bogota/bogota_o3_master_dataset.parquet (Bogota, daily)
Rows kept: those with all of ['pm25', 'o3', 'no2', 'no', 'co', 'temp_2m', 'rh_2m', 'blh_mean', 'radiation_mean', 'fire_eq1'] present.
N = 2917; dates 2014-01-03 to 2022-12-31.
Raw corr(pm25, o3) = 0.4926  (95% CI 0.465 to 0.520).
Method: residual-on-residual OLS partial correlation with intercept. Confidence intervals (ci_lo / ci_hi) are 95 % percentile
intervals of a moving-block bootstrap (30-day blocks, 2000 replicates, seed 1); the attenuation intervals are paired
(same resamples for the raw and the controlled correlation). Fisher-z intervals (days treated as independent, too narrow given
lag-1 autocorrelation ~0.75) are kept as ci_*_fisher.
pm25_net = residual of pm25 ~ co (R2 of that regression on this sample = 0.366).
Tercile test: partial corr(pm25*radiation_mean [or pm25_net*radiation_mean], o3 | n